In [ ]:
# ── 01 ── project root ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, json, platform, subprocess, datetime as dt
from pathlib import Path

ROOT  = Path("/content/drive/MyDrive/BNFDM")
MODEL = "laya-ml"
RUN   = dt.datetime.now().strftime("%Y%m%d-%H%M")

for d in ["data/splits", "configs", "results", "cache/hf"]:
    (ROOT / d).mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(ROOT / "cache/hf")

def R(*p):
    q = ROOT / "results" / MODEL / Path(*p); q.parent.mkdir(parents=True, exist_ok=True); return q
def D(*p):
    q = ROOT / "data" / Path(*p); q.parent.mkdir(parents=True, exist_ok=True); return q

ENV = {"run": RUN, "model": MODEL, "python": platform.python_version()}
try:
    ENV["gpu"] = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"]).decode().strip()
except Exception:
    ENV["gpu"] = "cpu"
print(json.dumps(ENV, indent=2)); print("root:", ROOT)

Mounted at /content/drive
{
  "run": "20261001-0007",
  "model": "laya-ml",
  "python": "3.13.15",
  "gpu": "Tesla T4, 15360 MiB"
}
root: /content/drive/MyDrive/BNFDM


In [ ]:
# ── 02 ── only what is missing. Never -U a preinstalled Colab package. ──────
import sys, subprocess
subprocess.run(f"{sys.executable} -m pip -q install laya kagglehub", shell=True, check=False)

import laya, pandas as pd, numpy as np
print("laya", laya.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

laya 0.3.22 | pandas 2.2.3 | numpy 2.1.3


In [ ]:
# ── 03 ── bangla-bagdhara. JSONL throughout (preserves list columns). ───────
import os, json, unicodedata, numpy as np, pandas as pd, kagglehub

RAW = D("bagdhara_raw.jsonl")
if RAW.exists():
    df = pd.read_json(RAW, lines=True); print("cached:", df.shape)
else:
    p = kagglehub.dataset_download("sakhadib/bangla-bagdhara")
    fs = os.listdir(p)
    jl, js = [f for f in fs if f.endswith(".jsonl")], [f for f in fs if f.endswith(".json")]
    df = pd.read_json(os.path.join(p, (jl or js)[0]), lines=bool(jl))
    df.to_json(RAW, orient="records", lines=True, force_ascii=False)
    print("downloaded:", df.shape)

def nfc(x):
    if isinstance(x, str):  return unicodedata.normalize("NFC", x).strip()
    if isinstance(x, list): return [nfc(v) for v in x]
    return x
for c in ["idiom","literal_meaning","figurative_meaning_bn","figurative_meaning_en",
          "note","example_sentences_in_bangla","example_sentences_in_english",
          "alternative_idioms","similar_in_english","tags","usage_domain","history"]:
    if c in df.columns: df[c] = df[c].map(nfc)

def lst(v):  return v if isinstance(v, list) else ([] if pd.isna(v) else [v])
def one(v):
    L = lst(v); return L[0] if L else ""

df["ex_bn1"]  = df["example_sentences_in_bangla"].map(one)
df["alt"]     = df["alternative_idioms"].map(lst)
df["sim_en"]  = df["similar_in_english"].map(one)
df["tag_lc"]  = df["tags"].map(lambda v: [str(t).strip().lower() for t in lst(v)])

n0 = len(df)
dd = (df.sort_values("id").drop_duplicates(subset=["idiom"], keep="first")
        .reset_index(drop=True))
# every item needs a gold figurative gloss and a gold literal gloss
dd = dd[(dd.figurative_meaning_bn.str.len() > 0) &
        (dd.literal_meaning.str.len() > 0)].reset_index(drop=True)
print(f"dedupe + require both glosses: {n0} -> {len(dd)}")

from sklearn.model_selection import train_test_split
SEED = 20261001
dev, test = train_test_split(dd, test_size=0.5, random_state=SEED)
probe = dev.sample(400, random_state=SEED)

for name, part in [("probe_400",probe), ("dev",dev), ("test",test), ("full",dd)]:
    part.reset_index(drop=True).to_json(D("splits", f"{name}.jsonl"),
        orient="records", lines=True, force_ascii=False)
    print(f"{name:<10}{len(part):>7}")

json.dump({"n_raw": n0, "n_usable": len(dd), "seed": SEED,
           "n_with_alt": int((dd.alt.map(len) > 0).sum()),
           "n_with_sim_en": int((dd.sim_en.str.len() > 0).sum()),
           "n_with_example": int((dd.ex_bn1.str.len() > 0).sum())},
          open(D("splits", "meta.json"), "w"), indent=2)
print(json.load(open(D("splits","meta.json"))))

Using Colab cache for faster access to the 'bangla-bagdhara' dataset.
downloaded: (10359, 19)
dedupe + require both glosses: 10359 -> 8876
probe_400     400
dev          4438
test         4438
full         8876
{'n_raw': 10359, 'n_usable': 8876, 'seed': 20261001, 'n_with_alt': 2445, 'n_with_sim_en': 8761, 'n_with_example': 8876}


In [ ]:
# ── 04 ── build k-way meaning-choice questions with controlled distractors ──
import re, random, unicodedata, numpy as np, pandas as pd, json
from collections import defaultdict

K = 4                                   # never 11+: that is the clamped-temperature bucket
OPT_KEYS = ["A", "B", "C", "D", "E", "F"][:K]
PUNCT = "।,;:!?\"'()[]{}—–-"

def words(t):
    t = re.sub(f"[{re.escape(PUNCT)}]", " ", unicodedata.normalize("NFC", str(t)))
    return [w for w in t.split() if w]

full = pd.read_json(D("splits","full.jsonl"), lines=True)

# ── indexes for surface- and tag-matched distractors ──────────────────────
tok2ids, tag2ids = defaultdict(set), defaultdict(set)
for r in full.itertuples():
    for w in set(words(r.idiom)):
        if len(w) >= 2: tok2ids[w].add(r.id)
    for t in r.tag_lc: tag2ids[t].add(r.id)

MEAN = dict(zip(full.id, full.figurative_meaning_bn))
LIT  = dict(zip(full.id, full.literal_meaning))
ALL_IDS = list(full.id)

def _fill(pool, need, exclude, rng):
    pool = [i for i in pool if i not in exclude]
    rng.shuffle(pool)
    out = pool[:need]
    while len(out) < need:
        c = rng.choice(ALL_IDS)
        if c not in exclude and c not in out: out.append(c)
    return out

def distractors(row, family, rng):
    """Return K-1 distractor STRINGS for this row under `family`."""
    ex = {row.id}
    if family == "D-rand":
        return [MEAN[i] for i in _fill(list(ALL_IDS), K-1, ex, rng)]
    if family == "D-lit":
        return [LIT[row.id]] + [MEAN[i] for i in _fill(list(ALL_IDS), K-2, ex, rng)]
    if family == "D-surf":
        cand = set()
        for w in set(words(row.idiom)):
            if len(w) >= 2: cand |= tok2ids.get(w, set())
        return [MEAN[i] for i in _fill(list(cand), K-1, ex, rng)]
    if family == "D-sem":
        cand = set()
        for t in row.tag_lc: cand |= tag2ids.get(t, set())
        return [MEAN[i] for i in _fill(list(cand), K-1, ex, rng)]
    if family == "D-hard":
        surf = set()
        for w in set(words(row.idiom)):
            if len(w) >= 2: surf |= tok2ids.get(w, set())
        sem = set()
        for t in row.tag_lc: sem |= tag2ids.get(t, set())
        d = [LIT[row.id]]
        d += [MEAN[i] for i in _fill(list(surf), 1, ex, rng)]
        d += [MEAN[i] for i in _fill(list(sem), K-3, ex | set(), rng)]
        return d[:K-1]
    raise ValueError(family)

FAMILIES = ["D-rand", "D-lit", "D-surf", "D-sem", "D-hard"]

INSTR = "নিচের বাংলা বাগধারাটির প্রকৃত ভাবার্থ কোনটি?"

def make_question(row, family, rng, options=None, instr=INSTR):
    """Options shuffled per item; returns (question_dict, gold_key, option_texts)."""
    if options is None:
        options = [row.figurative_meaning_bn] + distractors(row, family, rng)
        options = list(dict.fromkeys(options))[:K]          # dedupe exact collisions
        while len(options) < K:
            c = MEAN[rng.choice(ALL_IDS)]
            if c not in options: options.append(c)
    gold_text = row.figurative_meaning_bn
    order = list(range(len(options))); rng.shuffle(order)
    texts = [options[i] for i in order]
    crit  = {k: t for k, t in zip(OPT_KEYS, texts)}
    gold_key = OPT_KEYS[texts.index(gold_text)]
    return ({"type": "choice", "instructions": instr, "criteria": crit},
            gold_key, texts)

json.dump({"k": K, "families": FAMILIES, "instructions": INSTR,
           "option_keys": OPT_KEYS, "seed": SEED},
          open(ROOT / "configs/task.json", "w"), ensure_ascii=False, indent=2)
print("families:", FAMILIES, "| k =", K)

# eyeball one
rng = random.Random(0)
q, gk, tx = make_question(full.iloc[0], "D-lit", rng)
print("\nidiom:", full.iloc[0].idiom, "| gold key:", gk)
for k, v in q["criteria"].items(): print(f"  {k}: {v[:70]}")

families: ['D-rand', 'D-lit', 'D-surf', 'D-sem', 'D-hard'] | k = 4

idiom: অ আ ক খ | gold key: C
  A: অন্তঃসারশূন্য ব্যক্তি
  B: অপচয়, অপব্যয়, নাশ
  C: বর্ণজ্ঞান; বর্ণমালা
  D: প্রথম বাংলা বর্ণমালার অক্ষরসমূহ (the first letters of the Bengali alph


In [ ]:
# ── 05 ── multilingual checkpoint, pinned. No Router. ───────────────────────
import laya, json, random, pandas as pd

agent = laya.load("convaiinnovations/laya", subfolder="multilingual", device="cuda")
print("cfg:", {k: agent.cfg.get(k) for k in ("max_len","head_max_len","amp_dtype")})

# K gold glosses must fit the option budget. Defaults are 1024 / 256; glosses
# run to ~46 tokens at p95, so 4 options can crowd 256. Give the head room.
agent.cfg["max_len"], agent.cfg["head_max_len"] = 1024, 384
print("adjusted:", {k: agent.cfg[k] for k in ("max_len","head_max_len")})

def ask(state, q, qid="q", **kw):
    return agent.predict(state, {qid: q}, **kw)

full = pd.read_json(D("splits","full.jsonl"), lines=True)
rng  = random.Random(1)
row  = full.iloc[0]
q, gk, tx = make_question(row, "D-lit", rng)
r = ask(row.idiom, q); a = r["answers"]["q"]

assert set(a["probabilities"]) == set(OPT_KEYS)
assert abs(sum(a["probabilities"].values()) - 1) < 1e-3
print("\npred", a["choice"], "| gold", gk,
      "| conf", round(a["answer_confidence"], 4))
print("usage:", r["usage"])
assert not r["usage"]["truncated"], "options truncated — raise head_max_len"

json.dump({**ENV, "cfg": {k: agent.cfg.get(k) for k in
           ("max_len","head_max_len","amp_dtype","temperature","temperature_by_options")}},
          open(R("env.json"), "w"), indent=2, default=str)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

cfg: {'max_len': 1024, 'head_max_len': 256, 'amp_dtype': 'bf16'}
adjusted: {'max_len': 1024, 'head_max_len': 384}

pred B | gold D | conf 0.5715
usage: {'input_tokens': 144, 'output_tokens': 0, 'state_tokens': 4, 'state_tokens_dropped': 0, 'truncated': False, 'truncated_questions': []}


In [ ]:
# ── 06 ── T4 = Turing: fp16, no bf16. ───────────────────────────────────────
import torch, time, pandas as pd, random

print("gpu", torch.cuda.get_device_name(0), "| cap",
      torch.cuda.get_device_capability(0), "| bf16", torch.cuda.is_bf16_supported())

full  = pd.read_json(D("splits","full.jsonl"), lines=True)
rng   = random.Random(2)
qfix, _, _ = make_question(full.iloc[0], "D-rand", rng)   # one shared question set
states = list(full.idiom.head(256))
agent.predict_batch(states[:16], {"q": qfix}, batch_size=16)

rows = []
for bs in (16, 32, 64, 128):
    torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize(); t0 = time.time()
    agent.predict_batch(states, {"q": qfix}, batch_size=bs)
    torch.cuda.synchronize()
    ms = (time.time() - t0) / len(states) * 1000
    pk = torch.cuda.max_memory_allocated() / 1e9
    rows.append({"batch_size": bs, "ms_per_item": ms, "peak_gb": pk})
    print(f"bs={bs:>3}  {ms:5.1f} ms/item  peak {pk:.2f} GB")

# single-call latency, which is what the per-item experiments actually pay
t0 = time.time()
for s in states[:50]: agent.predict(s, {"q": qfix})
SINGLE_MS = (time.time() - t0) / 50 * 1000
print(f"single predict: {SINGLE_MS:.1f} ms/item")

bench = pd.DataFrame(rows); bench["single_ms"] = SINGLE_MS
bench.to_csv(R("bench_batch.csv"), index=False)
BATCH = int(bench.loc[bench.ms_per_item.idxmin(), "batch_size"])
print("BATCH =", BATCH)

gpu Tesla T4 | cap (7, 5) | bf16 True
bs= 16    8.5 ms/item  peak 1.63 GB
bs= 32    3.4 ms/item  peak 1.69 GB
bs= 64    3.1 ms/item  peak 1.79 GB
bs=128    3.1 ms/item  peak 2.01 GB
single predict: 27.2 ms/item
BATCH = 128


In [ ]:
# ── 07 ── E7: all K! orderings of a fixed option set, per item ──────────────
import itertools, random, json, numpy as np, pandas as pd
from tqdm.auto import tqdm

probe = pd.read_json(D("splits","full.jsonl"), lines=True).sample(
    1000, random_state=7).reset_index(drop=True)
PERMS = list(itertools.permutations(range(K)))          # 24 for K=4
rng   = random.Random(7)
JL    = R("e7_order", "raw.jsonl")
done  = {json.loads(l)["id"] for l in open(JL, encoding="utf-8")} if JL.exists() else set()
f     = open(JL, "a", encoding="utf-8")

for row in tqdm(list(probe.itertuples()), desc="E7"):
    if row.id in done: continue
    opts = [row.figurative_meaning_bn] + distractors(row, "D-hard", rng)
    opts = list(dict.fromkeys(opts))[:K]
    if len(opts) < K: continue
    gold_text = row.figurative_meaning_bn
    for pi, perm in enumerate(PERMS):
        texts = [opts[i] for i in perm]
        crit  = {k: t for k, t in zip(OPT_KEYS, texts)}
        gk    = OPT_KEYS[texts.index(gold_text)]
        a = ask(row.idiom, {"type":"choice","instructions":INSTR,"criteria":crit})["answers"]["q"]
        f.write(json.dumps({"id": int(row.id), "perm_id": pi,
            "perm": "".join(map(str, perm)), "gold_key": gk, "pred": a["choice"],
            "correct": int(a["choice"] == gk),
            "gold_pos": OPT_KEYS.index(gk),
            "pred_pos": OPT_KEYS.index(a["choice"]),
            "conf": a["answer_confidence"],
            "p_gold": a["probabilities"][gk],
            **{f"p_{k}": a["probabilities"][k] for k in OPT_KEYS}}) + "\n")
    f.flush()
f.close()

E7 = pd.read_json(JL, lines=True).drop_duplicates(["id","perm_id"], keep="last")
E7.to_csv(R("e7_order","raw.csv"), index=False, encoding="utf-8")

per_item = E7.groupby("id").agg(
    n_distinct_pred=("pred","nunique"), acc=("correct","mean"),
    p_gold_mean=("p_gold","mean"), p_gold_std=("p_gold","std"),
    conf_range=("conf", lambda s: s.max()-s.min())).reset_index()
per_item["unstable"] = per_item.n_distinct_pred > 1
per_item.to_csv(R("e7_order","per_item.csv"), index=False)

# position bias: does accuracy depend on WHERE the gold option sits?
pos = E7.groupby("gold_pos").agg(n=("correct","size"), acc=("correct","mean"),
                                 p_gold=("p_gold","mean")).reset_index()
pos.to_csv(R("e7_order","by_gold_position.csv"), index=False)
sel = E7.pred_pos.value_counts(normalize=True).sort_index().rename("share").reset_index()
sel.columns = ["pred_pos","share"]; sel.to_csv(R("e7_order","selection_bias.csv"), index=False)

print(f"unstable items        : {per_item.unstable.mean():.1%}")
print(f"mean distinct answers : {per_item.n_distinct_pred.mean():.2f} of {len(PERMS)}")
print(f"mean sd of P(gold)    : {per_item.p_gold_std.mean():.4f}")
print("\naccuracy by gold position:\n", pos.to_string(index=False))
print("\nselection share by position:\n", sel.to_string(index=False))

E7:   0%|          | 0/1000 [00:00<?, ?it/s]

unstable items        : 100.0%
mean distinct answers : 3.82 of 24
mean sd of P(gold)    : 0.0955

accuracy by gold position:
  gold_pos    n      acc   p_gold
        0 5952 0.066364 0.157580
        1 5952 0.214718 0.221385
        2 5952 0.147177 0.192956
        3 5952 0.146841 0.199412

selection share by position:
  pred_pos    share
        0 0.188382
        1 0.327453
        2 0.259199
        3 0.224966


In [ ]:
# ── 08 ── E1: accuracy and Literal Attraction Rate across distractor families
import random, json, numpy as np, pandas as pd
from tqdm.auto import tqdm

full = pd.read_json(D("splits","full.jsonl"), lines=True)
samp = full                                            # every usable idiom
print(f"{len(samp)} items x {len(FAMILIES)} families = "
      f"{len(samp)*len(FAMILIES):,} calls  (~{len(samp)*len(FAMILIES)*SINGLE_MS/60000:.0f} min)")

JL   = R("e1_ladder", "raw.jsonl")
done = {(d["id"], d["family"]) for d in
        (json.loads(l) for l in open(JL, encoding="utf-8"))} if JL.exists() else set()
f    = open(JL, "a", encoding="utf-8")

for row in tqdm(list(samp.itertuples()), desc="E1"):
    rng = random.Random(1000 + int(row.id))          # reproducible per item
    for fam in FAMILIES:
        if (row.id, fam) in done: continue
        q, gk, texts = make_question(row, fam, rng)
        a = ask(row.idiom, q)["answers"]["q"]
        lit_key = None
        if fam in ("D-lit", "D-hard") and row.literal_meaning in texts:
            lit_key = OPT_KEYS[texts.index(row.literal_meaning)]
        f.write(json.dumps({
            "id": int(row.id), "family": fam, "idiom": row.idiom,
            "gold_key": gk, "pred": a["choice"], "correct": int(a["choice"] == gk),
            "lit_key": lit_key,
            "chose_literal": int(lit_key is not None and a["choice"] == lit_key),
            "p_gold": a["probabilities"][gk],
            "p_literal": a["probabilities"][lit_key] if lit_key else None,
            "conf": a["answer_confidence"], "entropy_conf": a["confidence"],
            "gold_pos": OPT_KEYS.index(gk),
            "frequency": row.frequency, "scape": row.scape,
            "hist_sig": int(row.historical_significance),
            "relig_sig": int(row.religious_significance),
            "n_words": len(words(row.idiom)),
            **{f"p_{k}": a["probabilities"][k] for k in OPT_KEYS}}, ensure_ascii=False) + "\n")
    f.flush()
f.close()

E1 = pd.read_json(JL, lines=True).drop_duplicates(["id","family"], keep="last")
E1.to_csv(R("e1_ladder","raw.csv"), index=False, encoding="utf-8")

def ece(conf, corr, bins=15):
    conf, corr = np.asarray(conf,float), np.asarray(corr,float)
    ed = np.linspace(0,1,bins+1); e = 0.
    for lo,hi in zip(ed[:-1],ed[1:]):
        m = (conf>lo)&(conf<=hi)
        if m.sum(): e += m.mean()*abs(corr[m].mean()-conf[m].mean())
    return float(e)

summ = (E1.groupby("family")
          .apply(lambda g: pd.Series({
              "n": len(g), "accuracy": g.correct.mean(),
              "chance": 1/K,
              "LAR": g.chose_literal.mean() if g.chose_literal.notna().any() else np.nan,
              "mean_p_gold": g.p_gold.mean(),
              "mean_p_literal": g.p_literal.mean(),
              "mean_conf": g.conf.mean(), "ece": ece(g.conf, g.correct)}))
          .reset_index())
summ.to_csv(R("e1_ladder","summary.csv"), index=False)

# stratified breakdowns — all from gold metadata, no annotation
for col in ["frequency","scape","hist_sig","relig_sig","n_words","gold_pos"]:
    (E1.groupby(["family",col])
       .agg(n=("correct","size"), accuracy=("correct","mean"),
            LAR=("chose_literal","mean"), p_gold=("p_gold","mean"))
       .reset_index().to_csv(R("e1_ladder", f"by_{col}.csv"), index=False))

print(summ.to_string(index=False))

8876 items x 5 families = 44,380 calls  (~20 min)


E1:   0%|          | 0/8876 [00:00<?, ?it/s]

family      n  accuracy  chance      LAR  mean_p_gold  mean_p_literal  mean_conf      ece
D-hard 8876.0  0.144209    0.25 0.630464     0.193568        0.458395   0.532615 0.388640
 D-lit 8876.0  0.150856    0.25 0.643646     0.195286        0.463702   0.535880 0.385024
D-rand 8876.0  0.367170    0.25 0.000000     0.301046             NaN   0.438307 0.071137
 D-sem 8876.0  0.332920    0.25 0.000000     0.285525             NaN   0.432067 0.100172
D-surf 8876.0  0.337652    0.25 0.000000     0.288532             NaN   0.437102 0.099450


/tmp/ipykernel_2001/2884869521.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


In [ ]:
# ── 09 ── E2: same gold meaning, different surface. Does the answer hold? ───
import random, json, numpy as np, pandas as pd
from tqdm.auto import tqdm

full = pd.read_json(D("splits","full.jsonl"), lines=True)

# alternative_idioms can carry JSON nulls / empty strings — drop them here so
# no None ever reaches predict(). Also drop variants identical to the canonical.
def clean_alts(v, canon):
    out = []
    for a in (v if isinstance(v, list) else []):
        if a is None: continue
        s = str(a).strip()
        if s and s != str(canon).strip() and s not in out:
            out.append(s)
    return out

full["alt_clean"] = [clean_alts(r.alt, r.idiom) for r in full.itertuples()]
pairs = full[full.alt_clean.map(len) > 0].copy()
print(f"{len(pairs)} items with usable alternative surface forms "
      f"({int(full.alt.map(lambda v: len(v) if isinstance(v, list) else 0).sum()
              - full.alt_clean.map(len).sum())} null/dup variants dropped)")

def jsd(p, q):
    p, q = np.asarray(p,float), np.asarray(q,float); m = (p+q)/2
    def kl(a,b):
        a = np.clip(a,1e-12,1); b = np.clip(b,1e-12,1); return float((a*np.log(a/b)).sum())
    return 0.5*kl(p,m) + 0.5*kl(q,m)

JL   = R("e2_invariance","raw.jsonl")
done = {json.loads(l)["id"] for l in open(JL, encoding="utf-8")} if JL.exists() else set()
f    = open(JL, "a", encoding="utf-8")

for row in tqdm(list(pairs.itertuples()), desc="E2"):
    if row.id in done: continue
    rng = random.Random(2000 + int(row.id))
    # one fixed option set; only the STATE changes between canonical and variant
    opts = [row.figurative_meaning_bn] + distractors(row, "D-hard", rng)
    opts = list(dict.fromkeys(opts))[:K]
    if len(opts) < K: continue
    gold_text = row.figurative_meaning_bn
    order = list(range(K)); rng.shuffle(order)
    texts = [opts[i] for i in order]
    q  = {"type":"choice","instructions":INSTR,
          "criteria":{k:t for k,t in zip(OPT_KEYS, texts)}}
    gk = OPT_KEYS[texts.index(gold_text)]

    a0 = ask(row.idiom, q)["answers"]["q"]
    p0 = [a0["probabilities"][k] for k in OPT_KEYS]
    for vi, alt in enumerate(row.alt_clean[:2]):
        if not isinstance(alt, str) or not alt.strip():      # belt and braces
            continue
        a1 = ask(alt, q)["answers"]["q"]
        p1 = [a1["probabilities"][k] for k in OPT_KEYS]
        f.write(json.dumps({"id": int(row.id), "variant_i": vi,
            "idiom": row.idiom, "alt": alt, "gold_key": gk,
            "pred_canon": a0["choice"], "pred_alt": a1["choice"],
            "agree": int(a0["choice"] == a1["choice"]),
            "correct_canon": int(a0["choice"] == gk),
            "correct_alt": int(a1["choice"] == gk),
            "jsd": jsd(p0, p1),
            "p_gold_canon": a0["probabilities"][gk],
            "p_gold_alt": a1["probabilities"][gk],
            "conf_canon": a0["answer_confidence"], "conf_alt": a1["answer_confidence"]},
            ensure_ascii=False) + "\n")
    f.flush()
f.close()

E2 = pd.read_json(JL, lines=True).drop_duplicates(["id","variant_i"], keep="last")
E2.to_csv(R("e2_invariance","raw.csv"), index=False, encoding="utf-8")
pd.DataFrame([{
    "n_pairs": len(E2), "agreement": E2.agree.mean(),
    "acc_canonical": E2.correct_canon.mean(), "acc_alt": E2.correct_alt.mean(),
    "both_correct": ((E2.correct_canon==1)&(E2.correct_alt==1)).mean(),
    "mean_jsd": E2.jsd.mean(), "median_jsd": E2.jsd.median(),
    "mean_dP_gold": (E2.p_gold_canon - E2.p_gold_alt).abs().mean(),
}]).to_csv(R("e2_invariance","summary.csv"), index=False)
print(pd.read_csv(R("e2_invariance","summary.csv")).to_string(index=False))

2443 items with usable alternative surface forms (3 null/dup variants dropped)


E2:   0%|          | 0/2443 [00:00<?, ?it/s]

 n_pairs  agreement  acc_canonical  acc_alt  both_correct  mean_jsd  median_jsd  mean_dP_gold
    3143   0.657016       0.138085 0.193446      0.085269  0.039326    0.014872      0.080241


In [ ]:
# ── 10 ── E3: perturb the idiom, hold the options fixed ─────────────────────
import random, json, numpy as np, pandas as pd
from tqdm.auto import tqdm

full = pd.read_json(D("splits","full.jsonl"), lines=True)
samp = full[full.idiom.map(lambda s: len(words(s)) >= 3)].reset_index(drop=True)
VOCAB = sorted({w for s in full.idiom for w in words(s) if len(w) >= 2})
print(f"{len(samp)} items (>=3 words) x 6 variants = {len(samp)*6:,} calls "
      f"(~{len(samp)*6*SINGLE_MS/60000:.0f} min)")

def variants(idiom, example, rng):
    w = words(idiom)
    sh = w[:];
    for _ in range(8):
        rng.shuffle(sh)
        if sh != w: break
    dl = w[:]; di = rng.randrange(len(w)); drop = dl.pop(di)
    sb = w[:]; si = rng.randrange(len(w)); sb[si] = rng.choice(VOCAB)
    rv = list(reversed(w))
    out = {"V0_intact": idiom, "V1_shuffle": " ".join(sh), "V2_reverse": " ".join(rv),
           "V3_delete1": " ".join(dl), "V4_substitute1": " ".join(sb)}
    out["V5_in_sentence"] = example if example else idiom
    return out, drop, di, si

JL   = R("e3_unit","raw.jsonl")
done = {json.loads(l)["id"] for l in open(JL, encoding="utf-8")} if JL.exists() else set()
f    = open(JL, "a", encoding="utf-8")

for row in tqdm(list(samp.itertuples()), desc="E3"):
    if row.id in done: continue
    rng = random.Random(3000 + int(row.id))
    opts = [row.figurative_meaning_bn] + distractors(row, "D-hard", rng)
    opts = list(dict.fromkeys(opts))[:K]
    if len(opts) < K: continue
    gold_text = row.figurative_meaning_bn
    order = list(range(K)); rng.shuffle(order)
    texts = [opts[i] for i in order]
    q  = {"type":"choice","instructions":INSTR,
          "criteria":{k:t for k,t in zip(OPT_KEYS, texts)}}
    gk = OPT_KEYS[texts.index(gold_text)]
    vs, dropped, di, si = variants(row.idiom, row.ex_bn1, rng)
    base_p = None
    for vname, vtext in vs.items():
        a = ask(vtext, q)["answers"]["q"]
        p = [a["probabilities"][k] for k in OPT_KEYS]
        if vname == "V0_intact": base_p = p
        f.write(json.dumps({"id": int(row.id), "variant": vname, "text": vtext,
            "idiom": row.idiom, "gold_key": gk, "pred": a["choice"],
            "correct": int(a["choice"] == gk), "p_gold": a["probabilities"][gk],
            "conf": a["answer_confidence"],
            "jsd_vs_intact": float(jsd(base_p, p)) if base_p else 0.0,
            "n_words": len(words(row.idiom)), "dropped_word": dropped,
            "delete_pos": di, "sub_pos": si}, ensure_ascii=False) + "\n")
    f.flush()
f.close()

E3 = pd.read_json(JL, lines=True).drop_duplicates(["id","variant"], keep="last")
E3.to_csv(R("e3_unit","raw.csv"), index=False, encoding="utf-8")

base = E3[E3.variant=="V0_intact"].set_index("id")
E3 = E3.join(base[["correct","p_gold"]].rename(
    columns={"correct":"correct0","p_gold":"p_gold0"}), on="id")
E3["kept_answer"] = (E3.pred == E3.groupby("id").pred.transform(
    lambda s: s.iloc[0])).astype(int)
(E3.groupby("variant").agg(
    n=("correct","size"), accuracy=("correct","mean"),
    p_gold=("p_gold","mean"), delta_p_gold=("p_gold", lambda s: s.mean()),
    jsd=("jsd_vs_intact","mean"), answer_retained=("kept_answer","mean"),
    conf=("conf","mean")).reset_index()
   .to_csv(R("e3_unit","summary.csv"), index=False))
(E3.groupby(["variant","n_words"]).agg(n=("correct","size"),
    accuracy=("correct","mean"), jsd=("jsd_vs_intact","mean")).reset_index()
   .to_csv(R("e3_unit","by_length.csv"), index=False))
print(pd.read_csv(R("e3_unit","summary.csv")).to_string(index=False))

1791 items (>=3 words) x 6 variants = 10,746 calls (~5 min)


E3:   0%|          | 0/1791 [00:00<?, ?it/s]

       variant    n  accuracy   p_gold  delta_p_gold      jsd  answer_retained     conf
     V0_intact 1778  0.156918 0.209544      0.209544 0.000000         1.000000 0.465406
    V1_shuffle 1778  0.167604 0.211255      0.211255 0.014763         0.742407 0.475109
    V2_reverse 1778  0.178290 0.214963      0.214963 0.016851         0.706412 0.469970
    V3_delete1 1778  0.158043 0.208624      0.208624 0.014378         0.747469 0.474178
V4_substitute1 1778  0.161980 0.211656      0.211656 0.020249         0.692351 0.467296
V5_in_sentence 1778  0.237345 0.241269      0.241269 0.045553         0.550619 0.511848


In [ ]:
# ── 11 ── E4: Bangla-multilingual vs English-english vs English-multilingual
import random, json, numpy as np, pandas as pd
from tqdm.auto import tqdm

full = pd.read_json(D("splits","full.jsonl"), lines=True)
en = full[(full.sim_en.str.len() > 0) &
          (full.figurative_meaning_en.str.len() > 0)].copy()
samp = en.reset_index(drop=True)
print(f"{len(samp)} items with gold English equivalent + English gloss "
      f"= {len(samp)*4:,} calls (~{len(samp)*4*SINGLE_MS/60000:.0f} min)")

agent_en = laya.load("convaiinnovations/laya", device="cuda")   # English root
agent_en.cfg["max_len"], agent_en.cfg["head_max_len"] = 512, 256
print("english cfg:", {k: agent_en.cfg[k] for k in ("max_len","head_max_len")})

MEAN_EN = dict(zip(full.id, full.figurative_meaning_en))
INSTR_EN = "What is the actual figurative meaning of this idiom?"

def ask_with(ag, state, q):
    return ag.predict(state, {"q": q})["answers"]["q"]

JL   = R("e4_crosslingual","raw.jsonl")
done = {json.loads(l)["id"] for l in open(JL, encoding="utf-8")} if JL.exists() else set()
f    = open(JL, "a", encoding="utf-8")

for row in tqdm(list(samp.itertuples()), desc="E4"):
    if row.id in done: continue
    rng = random.Random(4000 + int(row.id))
    ex = {row.id}
    pool = _fill(list(ALL_IDS), K-1, ex, rng)

    # Bangla arm
    tb = [row.figurative_meaning_bn] + [MEAN[i] for i in pool]
    ob = list(range(K)); rng.shuffle(ob); tb = [tb[i] for i in ob]
    qb = {"type":"choice","instructions":INSTR,
          "criteria":{k:t for k,t in zip(OPT_KEYS, tb)}}
    gb = OPT_KEYS[tb.index(row.figurative_meaning_bn)]

    # English arm, same distractor items, English glosses
    te = [row.figurative_meaning_en] + [MEAN_EN[i] for i in pool]
    oe = list(range(K)); rng.shuffle(oe); te = [te[i] for i in oe]
    qe = {"type":"choice","instructions":INSTR_EN,
          "criteria":{k:t for k,t in zip(OPT_KEYS, te)}}
    ge = OPT_KEYS[te.index(row.figurative_meaning_en)]

    arms = {
      "bn_ml":  (agent,    row.idiom,  qb, gb),   # Bangla idiom, multilingual ckpt
      "en_en":  (agent_en, row.sim_en, qe, ge),   # English equiv, English ckpt
      "en_ml":  (agent,    row.sim_en, qe, ge),   # English equiv, multilingual ckpt
      "bn_en":  (agent_en, row.idiom,  qb, gb),   # control: English ckpt on Bengali
    }
    rec = {"id": int(row.id), "idiom": row.idiom, "sim_en": row.sim_en}
    for name, (ag, st, q, gk) in arms.items():
        a = ask_with(ag, st, q)
        rec[f"{name}_pred"]    = a["choice"]
        rec[f"{name}_correct"] = int(a["choice"] == gk)
        rec[f"{name}_p_gold"]  = a["probabilities"][gk]
        rec[f"{name}_conf"]    = a["answer_confidence"]
    f.write(json.dumps(rec, ensure_ascii=False) + "\n"); f.flush()
f.close()

E4 = pd.read_json(JL, lines=True).drop_duplicates("id", keep="last")
E4.to_csv(R("e4_crosslingual","raw.csv"), index=False, encoding="utf-8")
arms = ["bn_ml","en_en","en_ml","bn_en"]
pd.DataFrame([{"arm": a, "accuracy": E4[f"{a}_correct"].mean(),
               "mean_p_gold": E4[f"{a}_p_gold"].mean(),
               "mean_conf": E4[f"{a}_conf"].mean()} for a in arms]
            ).to_csv(R("e4_crosslingual","summary.csv"), index=False)

# failure localisation: right in English, wrong in Bangla = representation gap
E4["cell"] = np.select(
    [(E4.bn_ml_correct==1)&(E4.en_ml_correct==1),
     (E4.bn_ml_correct==0)&(E4.en_ml_correct==1),
     (E4.bn_ml_correct==1)&(E4.en_ml_correct==0)],
    ["both_right","bangla_gap","english_gap"], default="both_wrong")
E4.cell.value_counts(normalize=True).rename("share").reset_index().to_csv(
    R("e4_crosslingual","localisation.csv"), index=False)
print(pd.read_csv(R("e4_crosslingual","summary.csv")).to_string(index=False))
print("\n", E4.cell.value_counts(normalize=True).to_string())
del agent_en; torch.cuda.empty_cache()

8761 items with gold English equivalent + English gloss = 35,044 calls (~16 min)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/laya/agent.py:1668: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(model_id_or_path, device=device, token=token, subfolder=subfolder, fast=fast,


english cfg: {'max_len': 512, 'head_max_len': 256}


E4:   0%|          | 0/8761 [00:00<?, ?it/s]

  arm  accuracy  mean_p_gold  mean_conf
bn_ml  0.371191     0.302146   0.438409
en_en  0.733250     0.548098   0.612959
en_ml  0.616939     0.477855   0.584249
bn_en  0.282730     0.261488   0.389916

 cell
bangla_gap     0.373702
both_wrong     0.255108
both_right     0.243237
english_gap    0.127953


In [ ]:
# ── 12 ── E5: does confidence know when the model was captured? ────────────
import numpy as np, pandas as pd, json
from scipy.optimize import minimize_scalar

E1 = pd.read_csv(R("e1_ladder","raw.csv"))
P  = [f"p_{k}" for k in OPT_KEYS]

def temper(probs, T):
    lg = np.log(np.clip(probs, 1e-12, 1)) / T
    e  = np.exp(lg - lg.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

def nll(T, probs, gold_ix):
    q = temper(probs, T)
    return float(-np.log(np.clip(q[np.arange(len(q)), gold_ix], 1e-12, 1)).mean())

rows, curves = [], []
for fam, g in E1.groupby("family"):
    pr = g[P].values
    gi = g.gold_key.map({k: i for i, k in enumerate(OPT_KEYS)}).values
    corr = g.correct.values
    conf = pr.max(axis=1)
    # fit T on a random half, evaluate on the other
    rs = np.random.RandomState(5); m = rs.rand(len(g)) < 0.5
    T = minimize_scalar(lambda t: nll(t, pr[m], gi[m]),
                        bounds=(0.25, 8.0), method="bounded").x
    q = temper(pr[~m], T); conf_T = q.max(axis=1)
    pred_T = np.array(OPT_KEYS)[q.argmax(axis=1)]
    corr_T = (pred_T == g.gold_key.values[~m]).astype(int)
    rows.append({"family": fam, "T_fitted": float(T),
                 "ece_raw": ece(conf[~m], corr[~m]), "ece_tempered": ece(conf_T, corr_T),
                 "acc_raw": corr[~m].mean(), "acc_tempered": corr_T.mean(),
                 "mean_conf_raw": conf[~m].mean(), "mean_conf_tempered": conf_T.mean()})
    # risk-coverage
    o = np.argsort(-conf[~m])
    cc = np.cumsum(corr[~m][o]) / np.arange(1, (~m).sum()+1)
    for i in range(0, len(o), max(1, len(o)//100)):
        curves.append({"family": fam, "coverage": (i+1)/len(o),
                       "accuracy": cc[i], "threshold": conf[~m][o][i]})

pd.DataFrame(rows).to_csv(R("e5_calibration","temperature.csv"), index=False)
pd.DataFrame(curves).to_csv(R("e5_calibration","risk_coverage.csv"), index=False)

# does confidence separate literal capture from correct answers?
lit = E1[E1.family.isin(["D-lit","D-hard"])].dropna(subset=["chose_literal"])
pd.DataFrame([{
    "group": g, "n": len(d), "mean_conf": d.conf.mean(), "median_conf": d.conf.median(),
    "mean_p_gold": d.p_gold.mean(), "mean_p_literal": d.p_literal.mean()}
    for g, d in [("correct", lit[lit.correct==1]),
                 ("chose_literal", lit[lit.chose_literal==1]),
                 ("other_wrong", lit[(lit.correct==0)&(lit.chose_literal==0)])]
]).to_csv(R("e5_calibration","confidence_by_outcome.csv"), index=False)

# reliability bins, long form
rel = []
for fam, g in E1.groupby("family"):
    conf, corr = g.conf.values, g.correct.values
    ed = np.linspace(0,1,11)
    for lo,hi in zip(ed[:-1],ed[1:]):
        m = (conf>lo)&(conf<=hi)
        rel.append({"family":fam,"bin_lo":lo,"bin_hi":hi,"n":int(m.sum()),
                    "mean_conf": float(conf[m].mean()) if m.sum() else np.nan,
                    "accuracy": float(corr[m].mean()) if m.sum() else np.nan})
pd.DataFrame(rel).to_csv(R("e5_calibration","reliability.csv"), index=False)
print(pd.read_csv(R("e5_calibration","temperature.csv")).to_string(index=False))
print("\n", pd.read_csv(R("e5_calibration","confidence_by_outcome.csv")).to_string(index=False))

family  T_fitted  ece_raw  ece_tempered  acc_raw  acc_tempered  mean_conf_raw  mean_conf_tempered
D-hard  7.999996 0.386458      0.140612 0.146787      0.146787       0.532514            0.287399
 D-lit  7.999996 0.382920      0.136001 0.151973      0.151973       0.534893            0.287974
D-rand  1.696853 0.081371      0.014920 0.362345      0.362345       0.437763            0.361109
 D-sem  2.212997 0.100943      0.010461 0.332807      0.332807       0.432565            0.332149
D-surf  2.423209 0.095777      0.014781 0.340474      0.340474       0.436251            0.326703

         group     n  mean_conf  median_conf  mean_p_gold  mean_p_literal
      correct  2619   0.431749       0.4083     0.431749        0.234032
chose_literal 11309   0.596255       0.5562     0.144470        0.596255
  other_wrong  3853   0.421463       0.3970     0.181844        0.218608


In [ ]:
# ── 13 ── E6: perturb the OPTION text, hold state and gold fixed ───────────
import random, json, numpy as np, pandas as pd
from tqdm.auto import tqdm

full = pd.read_json(D("splits","full.jsonl"), lines=True)
samp = full[full.figurative_meaning_en.str.len() > 0].reset_index(drop=True)
MEAN_EN = dict(zip(full.id, full.figurative_meaning_en))
print(f"{len(samp)} items x 5 option-forms = {len(samp)*5:,} calls "
      f"(~{len(samp)*5*SINGLE_MS/60000:.0f} min)")

def trunc(t, n=6):  return " ".join(words(t)[:n])
def scram(t, rng):
    w = words(t); rng.shuffle(w); return " ".join(w)

JL   = R("e6_options","raw.jsonl")
done = {json.loads(l)["id"] for l in open(JL, encoding="utf-8")} if JL.exists() else set()
f    = open(JL, "a", encoding="utf-8")

for row in tqdm(list(samp.itertuples()), desc="E6"):
    if row.id in done: continue
    rng  = random.Random(6000 + int(row.id))
    pool = _fill(list(ALL_IDS), K-1, {row.id}, rng)
    ids  = [row.id] + pool
    order = list(range(K)); rng.shuffle(order); ids = [ids[i] for i in order]
    gi   = ids.index(row.id); gk = OPT_KEYS[gi]

    forms = {
      "O0_full_bn":   [MEAN[i] for i in ids],
      "O1_trunc6":    [trunc(MEAN[i]) for i in ids],
      "O2_scrambled": [scram(MEAN[i], rng) for i in ids],
      "O3_english":   [MEAN_EN.get(i, MEAN[i]) for i in ids],
      "O4_labels":    [f"অর্থ {j+1}" for j in range(K)],      # semantics removed
    }
    base_p = None
    for fname, texts in forms.items():
        q = {"type":"choice","instructions":INSTR,
             "criteria":{k:t for k,t in zip(OPT_KEYS, texts)}}
        a = ask(row.idiom, q)["answers"]["q"]
        p = [a["probabilities"][k] for k in OPT_KEYS]
        if fname == "O0_full_bn": base_p = p
        f.write(json.dumps({"id": int(row.id), "form": fname, "gold_key": gk,
            "pred": a["choice"], "correct": int(a["choice"] == gk),
            "p_gold": a["probabilities"][gk], "conf": a["answer_confidence"],
            "jsd_vs_full": float(jsd(base_p, p)) if base_p else 0.0,
            "opt_tokens": sum(len(words(t)) for t in texts)}, ensure_ascii=False) + "\n")
    f.flush()
f.close()

E6 = pd.read_json(JL, lines=True).drop_duplicates(["id","form"], keep="last")
E6.to_csv(R("e6_options","raw.csv"), index=False, encoding="utf-8")
(E6.groupby("form").agg(n=("correct","size"), accuracy=("correct","mean"),
    p_gold=("p_gold","mean"), conf=("conf","mean"),
    jsd_vs_full=("jsd_vs_full","mean"), opt_tokens=("opt_tokens","mean"))
   .reset_index().to_csv(R("e6_options","summary.csv"), index=False))
print(pd.read_csv(R("e6_options","summary.csv")).to_string(index=False))

8876 items x 5 option-forms = 44,380 calls (~20 min)


E6:   0%|          | 0/8876 [00:00<?, ?it/s]

        form    n  accuracy   p_gold     conf  jsd_vs_full  opt_tokens
  O0_full_bn 8876  0.364917 0.301184 0.438262     0.000000   19.194570
   O1_trunc6 8876  0.361311 0.300769 0.439220     0.013743   16.049685
O2_scrambled 8876  0.356692 0.297718 0.441792     0.031284   19.194570
  O3_english 8876  0.346890 0.303042 0.499202     0.070819   32.998423
   O4_labels 8876  0.255746 0.251029 0.505963     0.096133    8.000000


In [ ]:
# ── 14 ── inventory ─────────────────────────────────────────────────────────
import json, hashlib
def sha(p):
    h = hashlib.sha256()
    with open(p,"rb") as fh:
        while (b := fh.read(1<<20)): h.update(b)
    return h.hexdigest()[:16]

arte = sorted([*(ROOT/"results"/MODEL).rglob("*.csv"),
               *(ROOT/"results"/MODEL).rglob("*.json*"),
               *(ROOT/"data").rglob("*.json*"),
               *(ROOT/"configs").rglob("*.json")])
json.dump({**ENV, "artifacts":[{"path": str(a.relative_to(ROOT)),
           "bytes": a.stat().st_size, "sha256_16": sha(a)} for a in arte]},
          open(R(f"manifest_{RUN}.json"),"w"), indent=2, default=str)
for a in arte: print(f"{a.stat().st_size:>10,}  {a.relative_to(ROOT)}")
print(f"\n{len(arte)} artefacts")

       321  configs/task.json
14,206,587  data/bagdhara_raw.jsonl
 7,220,329  data/splits/dev.jsonl
14,410,542  data/splits/full.jsonl
       133  data/splits/meta.json
   661,547  data/splits/probe_400.jsonl
 7,190,213  data/splits/test.jsonl
       250  results/laya-ml/bench_batch.csv
     1,409  results/laya-ml/e1_ladder/by_frequency.csv
     1,287  results/laya-ml/e1_ladder/by_gold_pos.csv
       644  results/laya-ml/e1_ladder/by_hist_sig.csv
     2,723  results/laya-ml/e1_ladder/by_n_words.csv
       666  results/laya-ml/e1_ladder/by_relig_sig.csv
       978  results/laya-ml/e1_ladder/by_scape.csv
 6,938,429  results/laya-ml/e1_ladder/raw.csv
17,636,423  results/laya-ml/e1_ladder/raw.jsonl
       641  results/laya-ml/e1_ladder/summary.csv
   455,127  results/laya-ml/e2_invariance/raw.csv
 1,047,974  results/laya-ml/e2_invariance/raw.jsonl
       228  results/laya-ml/e2_invariance/summary.csv
     2,385  results/laya-ml/e3_unit/by_length.csv
 2,096,758  results/laya-ml/e3_unit/raw.